In [0]:
patient_df = spark.read.table(
    "db_healthcare_kl.silver.patient_silver_dlt"
)

encounter_df = spark.read.table(
    "db_healthcare_kl.silver.encounter_silver_dlt"
)

condition_df = spark.read.table(
    "db_healthcare_kl.silver.condition_silver_dlt"
)

observation_df = spark.read.table(
    "db_healthcare_kl.silver.observation_silver_dlt"
)

medication_df = spark.read.table(
    "db_healthcare_kl.silver.medication_request_silver_dlt"
)

In [0]:
patient_df.printSchema()

In [0]:
from pyspark.sql.functions import col, when, year, to_date

In [0]:
person_df = patient_df.select(
    col("Patient_Id").alias("person_id"),

    when(col("Gender") == "male", 8507)
    .when(col("Gender") == "female", 8532)
    .otherwise(0)
    .alias("gender_concept_id"),

    year(to_date(col("Birth_date"))).alias("year_of_birth"),

    col("given_name"),
    col("family_name")
)

In [0]:
display(person_df)

In [0]:
encounter_df.printSchema()

In [0]:
visit_occurrence_df = encounter_df.select(
    col("Encounter_Id").alias("visit_occurrence_id"),
    col("Patient_Reference").alias("person_id"),
    to_date(col("Start_Date")).alias("visit_start_date"),
    to_date(col("End_Date")).alias("visit_end_date"),
    col("Status").alias("visit_source_value"),
    col("Class").alias("visit_type")
)

In [0]:
condition_df.printSchema()

In [0]:
condition_occurrence_df = condition_df.select(
    col("Condition_Id").alias("condition_occurrence_id"),
    col("Patient_Reference").alias("person_id"),
    col("Encounter_Reference").alias("visit_occurrence_id"),
    col("Condition_Code").alias("condition_source_value"),
    col("Condition_Name").alias("condition_name"),
    to_date(col("Recorded_Date")).alias("condition_start_date"),
    to_date(col("Onset_Date")).alias("condition_end_date"),
    col("Clinical_Status").alias("clinical_status"),
    col("Verification_Status").alias("verification_status")
)

In [0]:
observation_df.printSchema()

In [0]:
observation_omop_df = observation_df.select(
    col("Observation_Id").alias("observation_id"),
    col("Patient_Reference").alias("person_id"),
    col("Encounter_Reference").alias("visit_occurrence_id"),
    col("Observation_Name").alias("observation_source_value"),
    col("Observation_Date").alias("observation_date"),
    col("Value").alias("value_as_number"),
    col("Unit").alias("unit_source_value"),
    col("Status").alias("observation_status"),
    col("Issued_Date").alias("issued_date")
)

In [0]:
medication_df.printSchema()

In [0]:
drug_exposure_df = medication_df.select(
    col("MedicationRequest_Id").alias("drug_exposure_id"),
    col("Patient_Reference").alias("person_id"),
    col("Encounter_Reference").alias("visit_occurrence_id"),
    col("Medication_Code").alias("drug_source_value"),
    col("Medication_Name").alias("drug_name"),
    to_date(col("Authored_On")).alias("drug_exposure_start_date"),
    col("Status").alias("drug_status"),
    col("Intent").alias("drug_intent"),
    col("Requester_Reference").alias("provider_id"),
    col("Requester_Name").alias("provider_name")
)